# Flickr API 

## What are API's ?

![API](https://media.geeksforgeeks.org/wp-content/uploads/20230216170349/What-is-an-API.png)

An API (Application Programming Interface) is a set of rules that allows different software programs to talk to each other.

You can think of it like a menu in a restaurant: the menu lists the dishes you can order, along with a description. When you tell the waiter what you want, the kitchen (the system behind the API) prepares it and sends it back to you.

In the same way, an API defines what requests you can make and what responses you’ll get—for example, asking a weather API for today’s temperature or a Flickr API for recent photos.

APIs are used everywhere: they let your app get data from a website, send messages, control devices, or connect services together.

## What is Flickr?

![Flickr](https://www.flickrhelp.com/hc/article_attachments/4419907628308/unnamed__1_.png)  

Flickr is an online platform for sharing, storing, and discovering photos and videos. It was one of the first major social networks built around photography.

Users can:

- Upload their own images and organize them into albums.
- Add tags, titles, and descriptions to make photos searchable.
- Explore images from other users around the world.
- Comment, like, and join photography groups or communities.

Flickr is especially popular among photographers, artists, and researchers because it provides high-quality image hosting and rich metadata (like geolocation, upload time, and camera info).

A typical Flickr post can be viewed here: [link](https://www.flickr.com/photos/92959567@N00/227080829).  

It also offers a simple public API, which allows developers to access that photo data programmatically — for example, to download geotagged images, analyze tags, or visualize where and what people photograph.

The objective of this exercise is to create a table containing images from a specific area along with their associated information.  

To achieve this, we will use the "Python Flickr API" library to handle authentication. Once authenticated, you can utilize the methods detailed in the API documentation [here](https://www.flickr.com/services/api/).  

The API methods fall under different authentication levels, such as:  
- *Requires 'write' permission for authentication.*  
- *Requires 'read' permission for authentication.*  
- *Does not require authentication.*  

Focus only on the methods that do not require authentication and disregard those needing "write" or "read" permissions.



## Geo Search

### Flickr subscription and API key

To run the live Flickr searches in this tutorial, **each student needs a Flickr Pro subscription and their own API key**. Flickr currently restricts requests for new API keys to Pro subscribers. Sign in or create a Flickr account, subscribe to Pro, then visit the [Flickr API Keys page](https://www.flickr.com/services/api/keys/) to apply for a non-commercial API key for educational use. Applications may require review, so arrange this before the workshop. Keep your API key and API secret private: never publish them on GitHub or include them in submitted notebooks. See [Flickr's official API guidance](https://www.flickrhelp.com/hc/en-us/articles/4404070036884-Flickr-API).

### Preparation & First Searches

Install the Python libraries if they are not already available. In Jupyter or Colab, run `%pip install flickrapi pandas folium` in a separate code cell. You only need to install them once per environment.

**Entering your Flickr credentials:** Each API application has a **key** and a **secret**. The next cell prompts you to enter both. The `getpass()` function hides what you type, so neither credential needs to appear in the notebook's code or printed output. The values stay in memory while the notebook kernel is running. If you restart the kernel, you will need to enter them again.

Do not replace these prompts with your actual key or secret when saving or sharing the notebook.

In [ ]:
from getpass import getpass

api_key = getpass("Flickr API key: ")
api_secret = getpass("Flickr API secret: ")

Import the libraries we need: `flickrapi` to communicate with Flickr, `pandas` for tables, and `folium` to draw interactive maps. The `FlickrAPI()` object uses the key and secret entered above to connect to Flickr.

In [6]:
import flickrapi
import pandas as pd
import folium


#connect to flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format='parsed-json')


### Define search location and display map

Flickr lets us search for photographs taken within a geographical area. We define this area with a **bounding box**: a rectangle described by its northern, southern, western and eastern limits.

The coordinates below describe a small area in central London. Latitude measures position north or south; longitude measures position east or west. You can find coordinates for another location using [OpenStreetMap](https://www.openstreetmap.org/).

There are **two different coordinate orders** to remember:

| Where we use coordinates | Required order |
| --- | --- |
| **Folium**, for map positions | `[latitude, longitude]` |
| **Flickr**, for a bounding box | `west longitude, south latitude, east longitude, north latitude` |

First we draw the search rectangle and its centre point on a Folium map. This lets us check the area visually before requesting any photographs.

In [ ]:
# Centre of our search area, in latitude / longitude order
centre_latitude = 51.51235
centre_longitude = -0.11720

# Bounding box: the four limits of our search rectangle
lat_north = 51.51491
lat_south = 51.50893
long_west = -0.12167
long_east = -0.11171

# Start with just two photos per page so the response is easy to inspect
per_page = 2

# Display the search area on a map
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True
)

# Mark the centre
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6
).add_to(map_london)

# Folium rectangle corners use [latitude, longitude] order
folium.Rectangle(
    bounds=[[lat_south, long_west], [lat_north, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue"
).add_to(map_london)

map_london

### Search Flickr for photographs

The main API method is [`flickr.photos.search`](https://www.flickr.com/services/api/flickr.photos.search.html). We send it the area we want to search and ask Flickr to return photographs with geographical information.

| Argument | Purpose |
| --- | --- |
| `bbox` | Search rectangle: **west, south, east, north** |
| `per_page` | Maximum number of photographs returned on each results page |
| `page` | Which results page to retrieve, starting at 1 |
| `has_geo=1` | Require photographs with geographical information |
| `extras` | Request additional details such as coordinates, tags and dates |

Unlike Folium, Flickr expects the bounding box as a **comma-separated string**, with longitude first. We construct it from the four variables rather than typing the coordinates again. By requesting only two photographs initially, the returned data is easier to inspect.

In [ ]:
# Flickr bbox order: west longitude, south latitude, east longitude, north latitude
bbox = f"{long_west},{lat_south},{long_east},{lat_north}"

photos = flickr.photos.search(
    bbox=bbox,
    per_page=per_page,
    page=1,
    has_geo=1,
    extras="geo,description,tags,views,media,url_n,date_taken,owner_name"
)

print(photos)

### Parsing the results: JSON

The search result is stored in the variable `photos`. Flickr sends its response in **JSON** (JavaScript Object Notation), a widely used format for exchanging structured data through APIs. Because we connected to Flickr with `format="parsed-json"`, the `flickrapi` library has already converted that JSON response into familiar Python **dictionaries** and **lists**.

JSON objects use named keys, much like Python dictionaries. Arrays contain sequences of values, much like Python lists. They can be nested inside one another.

For example, this is a simple JSON object containing a list:

```json
{
  "city": "London",
  "locations": ["Westminster", "Soho", "Bloomsbury"],
  "has_photos": true,
  "note": null
}
```

Notice that JSON writes `true`, `false` and `null` in lowercase. Once converted into Python, they become `True`, `False` and `None`.

### Inspect the Flickr response

The response contains an outer dictionary, a nested dictionary under the key `"photos"`, and a list of individual photograph records under `"photo"`. Each photograph is another dictionary, with keys such as `"id"`, `"title"` and `"latitude"`.

The structure is roughly:

```text
photos                         Python dictionary
└── "photos"                  Dictionary with search results
    ├── "page"                Current page number
    ├── "pages"               Total number of pages
    └── "photo"               List of photograph dictionaries
        ├── [0]               First photograph
        │   ├── "id"          Flickr photograph ID
        │   └── "title"       Photograph title
        └── [1]               Second photograph
```

We can use Python's built-in `pprint` module (*pretty print*) to display the response with indentation. This is useful when investigating an unfamiliar API.

In [ ]:
import pprint

pprint.pprint(photos, compact=True)

### Access values in nested dictionaries and lists

We can access each level using square brackets. **Strings** select dictionary keys, while **integers** select positions in a list. Python list positions start at zero.

Begin with the outer dictionary and explore the nested result:

In [ ]:
print(photos.keys())
print(photos["photos"]["page"])
print(photos["photos"]["pages"])

The list of photographs is found at `photos["photos"]["photo"]`. Let us store it in a shorter variable so the expressions become easier to read.

In [ ]:
photo_list = photos["photos"]["photo"]

print("Photographs on this page:", len(photo_list))
pprint.pprint(photo_list, compact=True)

If Flickr returned at least one photograph, `photo_list[0]` selects the **first photograph dictionary**, and `["id"]` extracts its ID. We check that the list is not empty before accessing its first element.

In [ ]:
if photo_list:
    first_photo = photo_list[0]
    print("Photo ID:", first_photo["id"])
    print("Title:", first_photo.get("title", "No title"))
else:
    print("No photographs were returned for this search.")

Here `.get("title", "No title")` retrieves the title if it exists, and supplies a fallback if it does not. By contrast, accessing a missing key directly with square brackets raises a `KeyError`.

The important idea is that **an API response is a structured collection of values**, not yet a spreadsheet-like table. In the next section we will retrieve several pages and convert the individual photo dictionaries into a Pandas DataFrame.

### Getting all pages (Pagination)

Our first request returned just **two photographs**, because we set `per_page = 2`. Flickr may have many more matching photographs, but it divides the response into **pages**. This is called *pagination*.

The response we already saved as `photos` tells us how many matching photographs and result pages are available:

- `photos["photos"]["total"]`: total number of matching photographs.
- `photos["photos"]["pages"]`: number of pages of results.
- `photos["photos"]["page"]`: the page returned by our request.

We can request another page by changing the `page` argument. Later we will use a loop to retrieve several pages and combine their photographs into a table.

### Determine the number of pages

We can read the counts directly from our first API response; we do not need to make the same request again.

In [ ]:
# Use the response from our first search
total_pages = int(photos["photos"]["pages"])
total_photos = int(photos["photos"]["total"])

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")
print(f"Current page: {photos['photos']['page']}")

# Display one example if the search returned photographs
if photos["photos"]["photo"]:
    first_photo = photos["photos"]["photo"][0]
    print("First photo ID:", first_photo["id"])
    print("Title:", first_photo.get("title", "No title"))
else:
    print("No photographs found in this area.")

### Extract Page Information

We now define a function called `get_page()`. It requests one results page, loops through its photograph records, and converts them into a **Pandas DataFrame**.

Each Flickr photograph is a dictionary. We extract the fields we want into a new dictionary (`new_row`), then collect these dictionaries in a list called `rows`. Finally, `pd.DataFrame(rows)` turns the list into a table.

Some photographs may not include every optional field, particularly image URLs or descriptions. `.get()` lets us supply a fallback value instead of stopping with a `KeyError`.

The function retains the column names used by the map below. For now, we keep the photo ID and geographical coordinates alongside the descriptive information.

In [ ]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )

    photos_on_page = response["photos"]["photo"]
    rows = []

    for photo in photos_on_page:
        description = photo.get("description") or {}
        if isinstance(description, dict):
            description = description.get("_content", "")

        new_row = {
            "id": photo.get("id"),
            "server": photo.get("server"),
            "secret": photo.get("secret"),
            "title": photo.get("title", ""),
            "tags": photo.get("tags", ""),
            "views": photo.get("views"),
            "description": description,
            "date_taken": photo.get("datetaken"),
            "latitude": photo.get("latitude"),
            "longitude": photo.get("longitude"),
            "url_s": photo.get("url_s"),
            "owner_name": photo.get("ownername"),
            "owner": photo.get("owner"),
            "media": photo.get("media"),
        }
        rows.append(new_row)

    return pd.DataFrame(rows)


# Try the function on the first page
get_page(bbox, 1, per_page)

### Go through several pages

Now we use a `for` loop to fetch multiple pages. For this introductory example, we deliberately limit the download to **five pages**. Retrieving hundreds of pages can take time and produce many API requests.

The expression `min(total_pages, 5)` ensures we never request more pages than Flickr says exist. Because `range()` excludes its end value, we add `+ 1` to include the final page.

Rather than expanding a DataFrame with every request, we collect the page DataFrames in a list and combine them **once** using `pd.concat()`. The result is stored in `df`, ready for the map in the next section.

In [ ]:
pages_to_download = min(total_pages, 5)
all_pages = []

for page in range(1, pages_to_download + 1):
    print(f"Getting page {page} of {total_pages}")
    page_df = get_page(bbox, page, per_page)
    all_pages.append(page_df)

if all_pages:
    df = pd.concat(all_pages, ignore_index=True)
else:
    df = pd.DataFrame(columns=["latitude", "longitude"])

print(f"Photographs retrieved: {len(df)}")
df

In [ ]:
df.head(10)

### Save the results as CSV

We can now export the collected photographs, just as we did in the Pandas tutorial. `to_csv()` creates a conventional comma-separated file. The option `index=False` avoids adding an unnecessary DataFrame index column.

Descriptions sometimes contain line breaks. We replace those with spaces in a separate DataFrame so that text is easier to read in a spreadsheet.

In [ ]:
export_df = df.replace(r"\r?\n", " ", regex=True)

export_df.to_csv("london_photos.csv", index=False)

print(f"Saved {len(export_df)} photographs to london_photos.csv")

### Map Display
We display the location of our points on the map: 

In [ ]:
# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    # popup="{} pixels".format(radius),    
).add_to(map_london) 

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",  
).add_to(map_london)

# Add a circle marker for each photo
latitudes = df['latitude']
longitudes = df['longitude']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid", 
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_london)
  
map_london

**Colors**  

Ever wondered where color names like "orchid" or "cornflowerblue" come from? These are built-in Windows colors, and you can find the complete list [here](https://learn.microsoft.com/en-us/dotnet/api/system.windows.media.colors?view=windowsdesktop-9.0).

---

## Combined Script

This is the combined script: 

### Import libraries and enter credentials

The combined script is a self-contained version of the previous steps. If you run it independently, enter your Flickr key and secret when prompted. As before, `getpass()` keeps them out of the notebook source and outputs.

In [ ]:
from getpass import getpass
import flickrapi
import pandas as pd
import pprint
import folium

api_key = getpass("Flickr API key: ")
api_secret = getpass("Flickr API secret: ")

# Connect to Flickr
flickr = flickrapi.FlickrAPI(api_key, api_secret, format="parsed-json")

### Set search parameters

In [126]:
# Define the location (latitude and longitude) and search parameters
centre_latitude = 51.51239
centre_longitude = 0.00496

# Define the bounding box. Use Open Street Map to get the coordinates
lat_north = 51.51550
lat_south = 51.50941
long_west = 0.00024
long_east = 0.01015

# Format: bbox = "min_longitude,min_latitude,max_longitude,max_latitude" for Flickr search
bbox = str(long_west)+','+str(lat_south)+','+str(long_east)+','+str(lat_north)

# Define the number of photos to retrieve per page
per_page = 250

# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False,
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(radius),
).add_to(map_london)

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north, long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue",
).add_to(map_london)

map_london

### Get amount of pages and photos

In [127]:
photos = flickr.photos.search(bbox=bbox, per_page=per_page, page=1, has_geo=1)

total_pages = photos["photos"]["pages"]
total_photos = photos["photos"]["total"]

print(f"Total photos: {total_photos}")
print(f"Total pages: {total_pages}")

Total photos: 3841
Total pages: 16


### Extract information

In [128]:
def get_page(bbox, page, per_page):
    response = flickr.photos.search(
        bbox=bbox,
        per_page=per_page,
        page=page,
        has_geo=1,
        extras="geo,description,tags,views,media,url_s,date_taken,owner_name",
    )
    photos = response["photos"]["photo"]
    rows = []
    for photo in photos:
        new_row = {
            "id": photo["id"],
            "server": photo["server"],
            "secret": photo["secret"],
            "title": photo["title"],
            "tags": photo["tags"],
            "views": photo["views"],
            "description": photo["description"]["_content"],
            "date_taken": photo["datetaken"],
            "latitude": photo["latitude"],
            "longitude": photo["longitude"],
            "url_s": photo["url_s"],
            "owner": photo["owner"],
            "owner_name": photo["ownername"],
            "media": photo["media"],
        }
        rows.append(new_row)
    df = pd.DataFrame(
        rows,
        columns=[
            "id",
            "server",
            "secret",
            "title",
            "tags",
            "views",
            "description",
            "date_taken",
            "latitude",
            "longitude",
            "url_s",
            "owner_name",
            "owner",
            "media",
        ],
    )
    return df


df = pd.DataFrame()

# Use this code to get only the first 10 pages of photos
for page in range(1, 10):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}")

""" 
Use this code to get all the photos in the bounding box: 
for page in range(1, total_pages + 1):
    new_df = get_page(bbox, page, per_page)
    print(f"Getting page {page} of {total_pages}")
    df = pd.concat([df, new_df], ignore_index=True)
    print(f"Total photos so far: {len(df)}") 
"""

df.to_csv("london_photos.csv", index=False)

df

Getting page 1 of 16
Total photos so far: 250
Getting page 2 of 16
Total photos so far: 500
Getting page 3 of 16
Total photos so far: 750
Getting page 4 of 16
Total photos so far: 1000
Getting page 5 of 16
Total photos so far: 1250
Getting page 6 of 16
Total photos so far: 1500
Getting page 7 of 16
Total photos so far: 1750
Getting page 8 of 16
Total photos so far: 2000
Getting page 9 of 16
Total photos so far: 2250


,id,server,secret,title,tags,views,description,date_taken,latitude,longitude,url_s,owner_name,owner,media
0,54188656132,65535,9eb07f8473,Go-Ahead London MHV6,goahead london mhv6 bu16oyo volvo b5lh mcv evo...,485,Go-Ahead London MHV6 (BU16 OYO)\nVolvo B5LH/MC...,2024-12-07 13:17:20,51.514731,0.007992,https://live.staticflickr.com/65535/5418865613...,gbenviro200,33732381@N04,photo
1,54188288630,65535,1e94c66157,EastLondon-37556-YX60DXO-CanningTown-300124,yx60dxo enviro200 firstcapital dm44169 route30...,433,East London 37556 (YX60 DXO) \n\nADL Enviro 20...,2024-01-30 00:00:00,51.514291,0.008368,https://live.staticflickr.com/65535/5418828863...,Michael Wadman,33075566@N08,photo
2,54186949737,65535,fbf67379c4,EastLondon-47992-YJ12GVR-CanningTown-300124,yj12gvr ctplus optaresolo optare solo route309...,470,East London 47992 (YJ12 GVR) \n\nFormer CT Plu...,2024-01-30 00:00:03,51.514291,0.008368,https://live.staticflickr.com/65535/5418694973...,Michael Wadman,33075566@N08,photo
3,54188115424,65535,6b76e2d37b,EastLondon-64202-LF20XKP-CanningTown-300124,lf20xkp bydd8ur route323 eastlondonbus canning...,400,East London 64202 (LF20 XKP) \n\nBYD D8UR / AD...,2024-01-30 00:00:02,51.514291,0.008368,https://live.staticflickr.com/65535/5418811542...,Michael Wadman,33075566@N08,photo
4,54186949517,65535,a17c975acf,EastLondon-47985-YJ60PFE-CanningTown-300124,yj60pfe ctplus optaresolo optare solo route309...,309,East London 47985 (YJ60 PFE) \n\nFormer CT Plu...,2024-01-30 00:00:01,51.514291,0.008368,https://live.staticflickr.com/65535/5418694951...,Michael Wadman,33075566@N08,photo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2245,19669717713,270,d30ac971ea,"City Island, Orchard Place",roundtower towerhamlets trinitybuoywharf round...,892,Thousands of private residents will be driving...,2015-07-11 11:03:51,51.510772,0.005171,https://live.staticflickr.com/270/19669717713_...,diamond geezer,36101699310@N01,photo
2246,20261193506,459,b17fea2926,Stagecoach London 19795 on route 5 at Canning ...,bus transport publictransport stagecoach londo...,2204,This is the closest route 5 gets to Central Lo...,2015-08-03 13:57:59,51.515259,0.007756,https://live.staticflickr.com/459/20261193506_...,SW11simon,42037567@N08,photo
2247,20276112171,3788,146972c94d,City Island,roundtower towerhamlets roundtower2,1237,One day this red footbridge will be a new way ...,2015-08-02 13:49:56,51.513363,0.005578,https://live.staticflickr.com/3788/20276112171...,diamond geezer,36101699310@N01,photo
2248,19644544533,3774,b8596fbdbc,City Island,roundtower towerhamlets roundtower2,1081,"Former industrial peninsula, soon to be luxury...",2015-08-02 13:48:49,51.513363,0.005578,https://live.staticflickr.com/3774/19644544533...,diamond geezer,36101699310@N01,photo


### Map display

In [ ]:
# Create a map centered on the location
map_london = folium.Map(
    location=[centre_latitude, centre_longitude],
    tiles="Cartodb dark_matter",
    zoom_start=16,
    control_scale=True,
    zoom_control=False,
    dragging=False,
    scrollWheelZoom=False
)

# Add a circle marker to the map
folium.CircleMarker(
    location=[centre_latitude, centre_longitude],
    radius=2,
    color="cornflowerblue",
    stroke=False,
    fill=True,
    fill_opacity=0.6,
    opacity=1,
    popup="{} pixels".format(radius),    
).add_to(map_london) 

# Add a rectangle to the map
folium.Rectangle(
    bounds=[[lat_north , long_west], [lat_south, long_east]],
    fill=True,
    fill_opacity=0.05,
    weight=.5,
    color="cornflowerblue",  
).add_to(map_london)

# Add a circle marker for each photo
latitudes = df['latitude']
longitudes = df['longitude']

for latitude, longitude in  zip(latitudes,longitudes):
  coordinate = [latitude,longitude]
  radius = 1
  folium.CircleMarker(
    location=coordinate,
    radius=radius,
    stroke=False,
    fill=True,
    fillColor="orchid", 
    fill_opacity=0.3,
    opacity=0.3,
  ).add_to(map_london)
  
map_london

## What's Next?  

If you examine this [Flickr Image](https://www.flickr.com/photos/33075566@N08/54187832141), you'll notice additional information associated with the image, such as "Favorites" or "Comments," which were not captured during the initial search.  
To address this, you'll need to iterate through the list and include the missing entries. Refer to the documentation to determine the appropriate API calls required to retrieve this data.

You can also convert the image's capture date into a datetime object, which helps you better analyze and understand the time aspect.